# NB-segformer — confirm SegFormer + test gated fusion on it

The sweep found **segformer_mit (3.7M)** the best + smallest model (1 seed:
mean 0.750, SAR-only 0.666). This does two things in one pass, 3 seeds each:

1. **segformer_concat** — early-fusion SegFormer + curriculum + consistency (the confirmed model).
2. **segformer_gated** — same backbone + the **gate/FiLM front-end** (identical 3.7M budget).

Decision: if segformer_gated beats segformer_concat (esp. on SAR-only), the fusion
modules finally add value on a strong backbone -> **architecture contribution**.
If not, the SegFormer + training recipe is the paper (still your best + smallest model).

**Attach:** `lgmf_c` (v7), Sen1Floods11, `lgmf-index`. GPU + Internet on. ~3h (6 runs).


In [ ]:
import sys, os, glob, json, random, time, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v7"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
try:
    import segmentation_models_pytorch  # noqa
except ImportError:
    os.system("pip install -q segmentation-models-pytorch")
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.baselines import build_baseline, build_gated_unet
from lgmf.model import count_params
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__)
idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))

In [ ]:
SEEDS, EPOCHS = [42, 1, 2], 45
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"; regs = ["both", "s1_only", "s2_only"]
ARCH, ENC = "segformer", "mit_b0"

# name -> "kind" (concat or gated)
RUNS = [("segformer_concat", "concat"), ("segformer_gated", "gated")]

def make_cfg(name, seed):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size = EPOCHS, 3e-4, 8
    c.ema_decay, c.loss_type, c.consistency_weight = 0.99, "focal_tversky", 0.5
    c.num_workers = 0
    c.split_dir, c.ckpt_dir = idx, "/kaggle/working/ckpt_segf"
    c.s1_mean, c.s1_std = stats["s1_mean"], stats["s1_std"]
    c.s2_mean, c.s2_std = stats["s2_mean"], stats["s2_std"]
    c.seed, c.run_name = seed, name
    return c

def build(kind):
    if kind == "gated":
        return build_gated_unet(arch=ARCH, encoder=ENC, out_ch=8, film=True, gate=True, pretrained=True)
    return build_baseline(arch=ARCH, encoder=ENC, in_ch=8, pretrained=True)

def seed_all(sd):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False

def run_one(cfg, kind):
    seed_all(cfg.seed)
    tr = DataLoader(Sen1Floods11(cfg, f"{idx}/train_index.json", augment=True),
                    batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=pin, drop_last=True)
    va = DataLoader(Sen1Floods11(cfg, f"{idx}/val_index.json", augment=False),
                    batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    model = build(kind).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type=="cuda") else None
    ema = ModelEMA(model, cfg.ema_decay); em = ema.module
    best, bv = -1.0, None
    for epoch in range(cfg.epochs):
        train_one_epoch(model, tr, opt, cfg, epoch, device, scaler=scaler, ema=ema)
        sched.step()
        val = evaluate(em, va, cfg, device)
        sc = float(np.mean([val[r]["iou"] for r in regs]))
        if sc > best:
            best = sc
            bv = {r: {"iou": val[r]["iou"], "f1": val[r]["f1"],
                      "flood_only_iou": val[r].get("flood_only_iou")} for r in regs}
            tag = ({"gated": {"out_ch": 8, "film": True, "gate": True}} if kind == "gated"
                   else {"baseline": {"in_ch": 8, "arch": ARCH, "encoder": ENC}})
            D = os.path.join(cfg.ckpt_dir, cfg.run_name, f"seed{cfg.seed}"); os.makedirs(D, exist_ok=True)
            torch.save({"model": model.state_dict(), "ema": ema.module.state_dict(),
                        "val": bv, "cfg": dict(cfg.__dict__), **tag}, os.path.join(D, "best.pt"))
    return best, bv, count_params(model)

## Train both x 3 seeds

In [ ]:
results = {name: [] for name, _ in RUNS}
params = {}
t0 = time.time()
for name, kind in RUNS:
    for sd in SEEDS:
        D = os.path.join("/kaggle/working/ckpt_segf", name, f"seed{sd}")
        rj = os.path.join(D, "result.json")
        if os.path.exists(rj):
            rec = json.load(open(rj)); best, bv = rec["best"], rec["val"]; params[name] = rec.get("params")
            print(f"[skip] {name:16s} seed{sd} mean {best:.3f}")
        else:
            best, bv, p = run_one(make_cfg(name, sd), kind); params[name] = p
            os.makedirs(D, exist_ok=True)
            json.dump({"best": best, "val": bv, "params": p}, open(rj, "w"), indent=2, default=str)
            print(f"[done] {name:16s} seed{sd} mean {best:.3f}  both {bv['both']['iou']:.3f} "
                  f"s1 {bv['s1_only']['iou']:.3f} s2 {bv['s2_only']['iou']:.3f}  ({(time.time()-t0)/60:.0f} min)")
        results[name].append(bv)
print(f"\ndone in {(time.time()-t0)/60:.0f} min")

## Verdict

In [ ]:
def ms(name, reg): a = np.array([r[reg]["iou"] for r in results[name]]); return a.mean(), a.std()
print("=" * 78)
print(f"{'model':18s} {'params':>7s} {'both':>12s} {'s1_only':>12s} {'s2_only':>12s}")
print("-" * 78)
for name, _ in RUNS:
    b, s1, s2 = ms(name,'both'), ms(name,'s1_only'), ms(name,'s2_only')
    p = params.get(name) or 3.72
    print(f"{name:18s} {p:6.2f}M {b[0]:.3f}±{b[1]:.3f} {s1[0]:.3f}±{s1[1]:.3f} {s2[0]:.3f}±{s2[1]:.3f}")
print("=" * 78)
cb, gb = ms('segformer_concat','both'), ms('segformer_gated','both')
cs, gs = ms('segformer_concat','s1_only'), ms('segformer_gated','s1_only')
print(f"\ngated - concat:  both {gb[0]-cb[0]:+.3f}   SAR-only {gs[0]-cs[0]:+.3f}")
if gs[0] - cs[0] > max(cs[1], gs[1]) or gb[0] - cb[0] > max(cb[1], gb[1]):
    print("-> gated front-end ADDS VALUE on SegFormer: architecture contribution.")
else:
    print("-> gated ~ concat: SegFormer + curriculum + consistency is the paper (best + smallest model).")
print("\nEither way, segformer_concat is your confirmed headline model. On to NB4 (test set).")